In [1]:
def make_M_V(P, w):
    M = P @ np.diag(w) @ P.T
    return M, np.diag(P @ w) - M

In [15]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

def max_qp_exact(Q):
    N = Q.shape[0]

    model = gp.Model()
    model.Params.OutputFlag = 0

    # x_i ∈ {0,1}
    x = model.addVars(N, vtype=GRB.BINARY, name="x")

    # y_i = 2x_i - 1
    y = [2 * x[i] - 1 for i in range(N)]

    # objective = y^T Q y
    objective = gp.quicksum(
        Q[i, j] * y[i] * y[j]
        for i in range(N)
        for j in range(N)
    )
    
    constraint1 =gp.quicksum(y)>=-1
    constraint2 =gp.quicksum(y)<=1
    
    model.setObjective(objective, GRB.MAXIMIZE)
    model.addConstr(constraint1)
    model.addConstr(constraint2)

    model.optimize()
    assert model.Status == GRB.OPTIMAL, \
        f"Gurobi did not prove optimality. Status = {model.Status}"

    y_opt = np.array([
        2 * round(x[i].X) - 1
        for i in range(N)
    ])

    return y_opt, model.ObjVal

In [16]:
graphs=[1,2,3,4,5]
for g in graphs:
    graph_type="poisson_16_12_3"
    if g < 10:
        hypergraph_name = f"0{g}"
    else:
        hypergraph_name = f"{g}"
    print("Name: ", hypergraph_name)
    P = np.load(f'data/{graph_type}/P/{hypergraph_name}.npy')
    w = np.load(f'data/{graph_type}/w/{hypergraph_name}.npy')
    M, Q = make_M_V(P, w)
    optimal_y, optimal_value = max_qp_exact(Q)
    print("Optimal y:", optimal_y)
    print("Optimal objective:", optimal_value)
    print(sum(optimal_y))



Name:  01
Optimal y: [-1 -1 -1  1 -1 -1  1 -1  1  1  1  1]
Optimal objective: 15.031111111111109
0
Name:  02
Optimal y: [ 1 -1 -1  1 -1 -1 -1  1  1 -1  1  1]
Optimal objective: 14.904444444444444
0
Name:  03
Optimal y: [ 1 -1  1 -1 -1  1 -1  1 -1  1 -1  1]
Optimal objective: 14.750000000000004
0
Name:  04
Optimal y: [-1  1 -1  1 -1  1 -1 -1  1 -1  1  1]
Optimal objective: 14.935555555555554
0
Name:  05
Optimal y: [-1 -1 -1  1  1  1 -1 -1  1  1  1 -1]
Optimal objective: 15.0
0
